# Baysor outputs → AnnData (.h5ad) — Stroke MERSCOPE (Fan / CG)

Collects the per-region Baysor outputs written by `baysor_stroke_batch.ipynb` and turns each into an
`.h5ad`:

- builds the **cell × gene** count matrix from `segmentation.csv` (transcripts with a `cell` assignment),
- attaches per-cell metadata from `segmentation_cell_stats.csv` (centroids, area, n_transcripts, …) to `.obs`,
- stores the centroid in `.obsm['spatial']`,
- writes one `<sample>.h5ad` per region and an optional concatenated `stroke_all.h5ad`.

**Kernel:** `sc` (the conda env with scanpy / pandas / numpy).

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad

sc.settings.verbosity = 1

# Must match `output_root` + `param_tag` from baysor_stroke_batch.ipynb
seg_root  = Path("/Volumes/T7/Stroke_merscop_Fan_CG/baysor_segmentation")
param_tag = "m50_s4"

# Where the .h5ad files are written
out_dir = Path("/Volumes/T7/Stroke_merscop_Fan_CG/h5ad")
out_dir.mkdir(parents=True, exist_ok=True)

# The two non-stroke (spinal cord) regions are handled separately in
# `spinalcord_to_h5ad_and_cluster.ipynb` (-> sc_all.h5ad). Exclude them here so the
# final `stroke_all.h5ad` contains only the brain / stroke sections.
SC_SAMPLES = ["SC_uninjured_1", "SC_uninjured_2"]

assert seg_root.exists(), f"Segmentation root not found: {seg_root}"

## 1. Find finished regions

In [ ]:
def find_outputs(root: Path, tag: str, exclude=None):
    exclude = set(exclude or [])
    found = []
    for seg in sorted(root.glob(f"*/{tag}/segmentation_segmentation.csv")):
        if seg.name.startswith("._"):
            continue
        sample = seg.parent.parent.name           # <root>/<sample>/<tag>/segmentation.csv
        if sample in exclude:
            continue
        stats = seg.parent / "segmentation_cell_stats.csv"
        if seg.stat().st_size > 0 and stats.exists():
            found.append({"sample": sample, "seg": seg, "stats": stats})
    return found

outputs = find_outputs(seg_root, param_tag, exclude=SC_SAMPLES)
print(f"Found {len(outputs)} finished regions (spinal cord excluded):")
for o in outputs:
    print("  ", o["sample"])

## 2. Build one AnnData per region

Baysor's `segmentation.csv` is one row per transcript with a `cell` column (0 / empty = background noise,
which we drop). We cross-tabulate transcripts into a cell × gene matrix and align it with the cell-stats table.

In [3]:
def build_adata(sample: str, seg: Path, stats: Path) -> ad.AnnData:
    tx = pd.read_csv(seg)
    tx = tx[tx["cell"].notna()]
    tx["cell"] = tx["cell"].astype(str)
    # Drop the background label (Baysor uses 0 / empty for noise transcripts)
    tx = tx[~tx["cell"].isin(["0", "", "nan"])]
    tx["gene"] = tx["gene"].astype(str)

    counts = pd.crosstab(tx["cell"], tx["gene"])

    st = pd.read_csv(stats)
    st["cell"] = st["cell"].astype(str)
    st = st.set_index("cell").reindex(counts.index)

    adata = ad.AnnData(
        X=counts.values.astype(np.float32),
        obs=st,
        var=pd.DataFrame(index=counts.columns),
    )
    adata.obs_names = [f"{sample}_{c}" for c in counts.index]
    adata.obs["sample"] = sample

    # Spatial centroid -> .obsm['spatial'] (Baysor stats name the centroid x/y)
    xcol = next((c for c in ("x", "centroid_x", "global_x") if c in adata.obs), None)
    ycol = next((c for c in ("y", "centroid_y", "global_y") if c in adata.obs), None)
    if xcol and ycol:
        adata.obsm["spatial"] = adata.obs[[xcol, ycol]].to_numpy(dtype=float)
    return adata


adatas = {}
for o in outputs:
    a = build_adata(o["sample"], o["seg"], o["stats"])
    adatas[o["sample"]] = a
    out_path = out_dir / f"{o['sample']}.h5ad"
    a.write_h5ad(out_path)
    print(f"{o['sample']:40s}  cells={a.n_obs:6d}  genes={a.n_vars:4d}  ->  {out_path.name}")

1005_glast_dmcao_5d_1                     cells= 60283  genes= 500  ->  1005_glast_dmcao_5d_1.h5ad


1005_glast_dmcao_5d_2                     cells= 80060  genes= 500  ->  1005_glast_dmcao_5d_2.h5ad


1149_glast_dmcao_clp_7d_1                 cells=136505  genes= 500  ->  1149_glast_dmcao_clp_7d_1.h5ad


1149_glast_dmcao_clp_7d_2                 cells=151686  genes= 500  ->  1149_glast_dmcao_clp_7d_2.h5ad


1149_glast_dmcao_clp_7d_3                 cells=124845  genes= 500  ->  1149_glast_dmcao_clp_7d_3.h5ad


...

B470_col1a1_dmcao_14d                     cells= 22668  genes= 500  ->  B470_col1a1_dmcao_14d.h5ad


B647_col1a1_dmcao_5d                      cells= 71484  genes= 500  ->  B647_col1a1_dmcao_5d.h5ad


B751_col1a1_tmcao_5d                      cells= 23429  genes= 500  ->  B751_col1a1_tmcao_5d.h5ad


SC_uninjured_1                            cells= 61450  genes= 500  ->  SC_uninjured_1.h5ad


SC_uninjured_2                            cells= 53881  genes= 500  ->  SC_uninjured_2.h5ad


## 3. (Optional) concatenate all regions into one object

Outer join on genes so panels with small differences still align; `sample` in `.obs` keeps regions separable.

In [4]:
if adatas:
    combined = ad.concat(adatas, join="outer", label="sample", index_unique=None)
    combined.X = np.nan_to_num(combined.X)
    combined_path = out_dir / "stroke_all.h5ad"
    combined.write_h5ad(combined_path)
    print(combined)
    print("\nWrote", combined_path)
else:
    print("No finished regions yet — run baysor_stroke_batch.ipynb first.")

AnnData object with n_obs × n_vars = 3776481 × 500
    obs: 'x', 'y', 'z', 'cluster', 'n_transcripts', 'density', 'elongation', 'area', 'avg_confidence', 'avg_assignment_confidence', 'max_cluster_frac', 'lifespan', 'sample'
    obsm: 'spatial'

Wrote /Volumes/T7/Stroke_merscop_Fan_CG/h5ad/stroke_all.h5ad
